# 工具访问 — InjectedState 与 InjectedStore

工具可以通过依赖注入机制自动获取 Agent 状态（State）和持久化存储（Store）中的信息。

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

环境变量已加载


## InjectedState——在工具中访问 Agent 状态

工具通过 `InjectedState` 可以直接读取 Agent 的完整状态，如消息历史、用户已确认的信息等。

In [3]:
from typing import Annotated, Any
from langchain.tools import tool, InjectedState

@tool
def remember_preference(
    preference: str,
    state: Annotated[dict[str, Any], InjectedState],
) -> str:
    """记住用户的偏好设置。

    Args:
        preference: 用户的偏好内容
        state: 系统自动注入的当前 Agent 状态
    """
    messages = state.get("messages", [])
    previous_prefs = state.get("user_preferences", "无")
    return (
        f"已记住偏好: {preference}。"
        f"(当前对话共 {len(messages)} 条消息，之前偏好: {previous_prefs})"
    )

# InjectedState 由框架自动注入
# 直接调用时需手动传入 state（在 Agent 中由框架自动注入）
result = remember_preference.invoke({
    "preference": "暗色主题",
    "state": {"messages": [], "user_preferences": "亮色主题"},
})
print(result)

已记住偏好: 暗色主题。(当前对话共 0 条消息，之前偏好: 亮色主题)


### 在 Agent 中使用 InjectedState

In [5]:
from typing import Annotated, Any
from langchain.tools import tool, InjectedState

@tool
def conversation_stats(
    state: Annotated[dict[str, Any], InjectedState],
) -> str:
    """获取当前对话的统计信息。
    不需要任何参数，统计信息从当前状态中自动读取。
    """
    messages = state.get("messages", [])
    human_msgs = [m for m in messages if m.type == "human"]
    ai_msgs = [m for m in messages if m.type == "ai"]
    tool_msgs = [m for m in messages if m.type == "tool"]
    return (
        f"共 {len(messages)} 条 | 用户 {len(human_msgs)} | "
        f"AI {len(ai_msgs)} | 工具 {len(tool_msgs)}"
    )

print(conversation_stats.invoke({"state": {"messages": []}}))

共 0 条 | 用户 0 | AI 0 | 工具 0


## InjectedStore——访问持久化存储

State 是对话级别的，对话结束就没了。Store 是跨会话持久化存储，适合保存用户偏好、学习进度等长期信息。

In [7]:
from typing import Annotated
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, InjectedStore

# 创建持久化存储
store = InMemoryStore()
store.put(("users", "user_001"), "profile", {
    "data": {
        "name": "小明",
        "level": "入门",
        "completed_courses": ["HTML 基础教程"]
    }
})

@tool
def get_user_profile(
    store: Annotated[BaseStore, InjectedStore()],
) -> str:
    """获取当前用户的学习档案信息。"""
    item = store.get(("users", "user_001"), "profile")
    if item is None:
        return "未找到用户档案"
    p = item.value["data"]
    return f"姓名={p['name']}，水平={p['level']}，已完成={', '.join(p['completed_courses'])}"

@tool
def save_course_progress(
    course_name: str,
    store: Annotated[BaseStore, InjectedStore()],
) -> str:
    """保存用户的学习进度。

    Args:
        course_name: 已完成的课程名称
    """
    item = store.get(("users", "user_001"), "profile")
    profile = item.value["data"] if item else {"name": "小明", "level": "入门", "completed_courses": []}
    if course_name not in profile["completed_courses"]:
        profile["completed_courses"].append(course_name)
    store.put(("users", "user_001"), "profile", {"data": profile})
    return f"已更新！共 {len(profile['completed_courses'])} 门：{', '.join(profile['completed_courses'])}"

print(get_user_profile.invoke({"store": store}))
print(save_course_progress.invoke({"course_name": "Python3 基础教程", "store": store}))
print(get_user_profile.invoke({"store": store}))

姓名=小明，水平=入门，已完成=HTML 基础教程
已更新！共 2 门：HTML 基础教程, Python3 基础教程
姓名=小明，水平=入门，已完成=HTML 基础教程, Python3 基础教程


## 在 Agent 中结合 Store

将 Store 传给 `create_agent()`，Agent 中的所有工具都能通过 `InjectedStore` 访问它。

In [8]:
from typing import Annotated
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, InjectedStore
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

store = InMemoryStore()
store.put(("runoob", "courses"), "catalog", {
    "data": {
        "Python3 基础教程": {"price": "免费", "duration": "20小时"},
        "Python 数据分析": {"price": "会员", "duration": "30小时"},
    }
})

@tool
def query_course_price(
    course_name: str,
    store: Annotated[BaseStore, InjectedStore()],
) -> str:
    """查询课程价格信息。

    Args:
        course_name: 课程名称
    """
    item = store.get(("runoob", "courses"), "catalog")
    catalog = item.value["data"] if item else {}
    if course_name in catalog:
        info = catalog[course_name]
        return f"《{course_name}》- {info['price']}，{info['duration']}"
    return f"未找到《{course_name}》"

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model,
    tools=[query_course_price],
    store=store,
    system_prompt="你是菜鸟教程 RUNOOB 的课程顾问。",
)

result = agent.invoke({"messages": [HumanMessage(content="Python3 基础教程多少钱？")]})
print(result["messages"][-1].content)

好消息！**《Python3 基础教程》是完全免费的！** 🎉

以下是详细信息：

- 📚 **课程名称**：Python3 基础教程
- 💰 **价格**：**免费**
- ⏱ **时长**：20 小时

您无需支付任何费用即可学习这门课程，非常适合想要入门 Python 编程的朋友！如果您有任何其他问题或想了解其他课程的信息，随时问我哦~ 😊


## InjectedState vs InjectedStore

| 维度 | InjectedState | InjectedStore |
| --- | --- | --- |
| 作用域 | 当前对话 | 跨会话 |
| 生命周期 | 对话结束即消失 | 持久化 |
| 典型用途 | 消息历史、中间结果 | 用户偏好、学习进度 |
| 数据组织 | 扁平字典 | 命名空间 + 键的层级结构 |

**术语：**

- **InjectedState**：注入 Agent 当前状态，自动注入
- **InjectedStore**：注入持久化存储（需要括号），跨会话共享
- **InjectedToolArg**：通用注入标记基类
- **InMemoryStore**：内存存储，用于测试；生产环境用数据库实现